# XGBoost: trzy strategie brakow danych dla 13 wskaznikow
Mediana + flagi, natywne NaN oraz strategia mieszana.
W kazdym przypadku ten sam podzial, walidacja i parametry modelu.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_recall_curve, average_precision_score, roc_auc_score,
    precision_score, recall_score, fbeta_score, accuracy_score, confusion_matrix
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
PLIK = Path('5year.arff')
if not PLIK.exists():
    PLIK = Path('/mnt/data/5year.arff')
if not PLIK.exists():
    raise FileNotFoundError('Umiesc 5year.arff w folderze z notatnikiem.')

NUMERY_CECH = [1, 2, 3, 5, 6, 24, 29, 34, 35, 46, 56, 58, 62]
CECHY = [f'Attr{i}' for i in NUMERY_CECH]
MEDIANA_W_HYBRYDZIE = ['Attr1', 'Attr2', 'Attr3', 'Attr5', 'Attr6', 'Attr29', 'Attr35']
NAN_W_HYBRYDZIE = ['Attr24', 'Attr34', 'Attr46']

## 1. Dane i podzial
Duplikaty usuwamy na pelnych 64 cechach plus etykiecie, przed selekcja 13 kolumn.


In [2]:
rekordy, _ = arff.loadarff(str(PLIK))
kolumny = [c for c in rekordy.dtype.names if c != 'class']
dane = pd.DataFrame({c: rekordy[c].astype(float) for c in kolumny})
dane['class'] = [int(v.decode('ascii') if isinstance(v, bytes) else v)
                 for v in rekordy['class']]
dane = dane.drop_duplicates().reset_index(drop=True)
y = dane.pop('class').astype(int)
X = dane.loc[:, CECHY].copy()

print('Liczba firm:', len(X), '| bankructwa:', int(y.sum()))
print('\nBraki w wybranych wskaznikach:')
print(pd.DataFrame({
    'wskaznik': [f'X{i}' for i in NUMERY_CECH],
    'braki': X.isna().sum().to_numpy(),
    'procent': (100 * X.isna().mean()).round(2).to_numpy(),
}).to_string(index=False))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f'\nTrening: {len(X_train)}; test: {len(X_test)}; bankructwa test: {int(y_test.sum())}')

Liczba firm: 5850 | bankructwa: 408

Braki w wybranych wskaznikach:
wskaznik  braki  procent
      X1      3     0.05
      X2      3     0.05
      X3      3     0.05
      X5     11     0.19
      X6      3     0.05
     X24    135     2.31
     X29      3     0.05
     X34     18     0.31
     X35      3     0.05
     X46     21     0.36
     X56      0     0.00
     X58      0     0.00
     X62      0     0.00

Trening: 4680; test: 1170; bankructwa test: 82


## 2. Przetwarzanie w pipeline, bez wycieku informacji
Granice przyciecia i mediany obliczane sa TYLKO na zbiorze treningowym
(oddzielnie w kazdym foldzie walidacji). Braki w wybranych cechach
pozostaja NaN i sa obslugiwane przez XGBoost.


In [3]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        a = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(a, self.lower, axis=0)
        self.upper_ = np.nanquantile(a, self.upper, axis=0)
        return self

    def transform(self, X):
        return pd.DataFrame(
            np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_),
            index=X.index, columns=X.columns
        )


class HybrydowaImputacja(BaseEstimator, TransformerMixin):
    """Mediany + flagi tylko dla wskazanych kolumn; inne NaN zostaja."""

    def __init__(self, kolumny_medianowe=None, dodaj_flagi=True):
        self.kolumny_medianowe = kolumny_medianowe
        self.dodaj_flagi = dodaj_flagi

    def fit(self, X, y=None):
        self.kolumny_ = list(X.columns)
        self.kolumny_medianowe_ = list(self.kolumny_medianowe or [])
        nieznane = set(self.kolumny_medianowe_) - set(self.kolumny_)
        if nieznane:
            raise ValueError(f'Nieznane kolumny: {nieznane}')
        self.mediany_ = X[self.kolumny_medianowe_].median()
        if self.mediany_.isna().any():
            raise ValueError('Nie mozna obliczyc mediany: wybrana kolumna jest cala pusta.')
        # Flagi tylko dla kolumn, ktore w tej czesci treningu maja braki.
        self.flagowane_ = [c for c in self.kolumny_medianowe_
                           if self.dodaj_flagi and X[c].isna().any()]
        return self

    def transform(self, X):
        X = X.loc[:, self.kolumny_].copy()
        for col in self.flagowane_:
            X[f'brak_{col}'] = X[col].isna().astype(int)
        X[self.kolumny_medianowe_] = (
            X[self.kolumny_medianowe_].fillna(self.mediany_)
        )
        return X


PARAMETRY_XGB = dict(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    scale_pos_weight=1,
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=2,
)


def zbuduj_model(wariant):
    kroki = [('winsoryzacja', QuantileClipper(0.01, 0.99))]
    if wariant == 'Mediana + flagi':
        kroki.append(('imputacja', SimpleImputer(
            strategy='median', add_indicator=True
        )))
    elif wariant == 'Hybryda: mediana + NaN':
        kroki.append(('imputacja', HybrydowaImputacja(
            kolumny_medianowe=MEDIANA_W_HYBRYDZIE,
            dodaj_flagi=True
        )))
    elif wariant != 'NaN (XGBoost)':
        raise ValueError('Nieznany wariant: ' + str(wariant))

    kroki.append(('xgboost', XGBClassifier(**PARAMETRY_XGB)))
    return Pipeline(kroki)


def wybierz_prog_f2(y_true, prawdopodobienstwa):
    precision, recall, progi = precision_recall_curve(y_true, prawdopodobienstwa)
    f2 = (5 * precision[:-1] * recall[:-1]) / (
        4 * precision[:-1] + recall[:-1] + 1e-12
    )
    idx = int(np.argmax(f2))
    return float(progi[idx]), float(f2[idx])

## 3. Sprawdzenie, gdzie pozostaly NaN po przetwarzaniu


In [4]:
model_kontrolny = zbuduj_model('Hybryda: mediana + NaN')
X_hybryda = model_kontrolny.named_steps['imputacja'].fit_transform(
    model_kontrolny.named_steps['winsoryzacja'].fit_transform(X_train)
)
print('\nBraki po imputacji hybrydowej (trening):')
print(X_hybryda.isna().sum()[lambda s: s > 0].to_string())
print('Flagi dodane:', [c for c in X_hybryda.columns if c.startswith('brak_')])


Braki po imputacji hybrydowej (trening):
Attr24    107
Attr34     13
Attr46     15
Flagi dodane: ['brak_Attr1', 'brak_Attr2', 'brak_Attr3', 'brak_Attr5', 'brak_Attr6', 'brak_Attr29', 'brak_Attr35']


## 4. Walidacja OOF i ocena na odlozonym tescie
Prog F2 wybieramy z predykcji OOF na treningu.
Zbior testowy sluzy WYLACZNIE do raportowania wynikow.


In [5]:
warianty = ['Mediana + flagi', 'NaN (XGBoost)', 'Hybryda: mediana + NaN']
wyniki = []
modele = {}
progi_wybrane = {}

for wariant in warianty:
    model = zbuduj_model(wariant)
    p_oof = cross_val_predict(
        model, X_train, y_train, cv=cv,
        method='predict_proba', n_jobs=1
    )[:, 1]
    prog, f2_oof = wybierz_prog_f2(y_train, p_oof)
    model.fit(X_train, y_train)
    p_test = model.predict_proba(X_test)[:, 1]
    pred = (p_test >= prog).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()

    wyniki.append({
        'Wariant': wariant,
        'Prog_F2_OOF': prog,
        'AP_OOF': average_precision_score(y_train, p_oof),
        'F2_OOF_dobrane': f2_oof,
        'AP_test': average_precision_score(y_test, p_test),
        'ROC_AUC_test': roc_auc_score(y_test, p_test),
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F2': fbeta_score(y_test, pred, beta=2, zero_division=0),
        'TP_wykryte': int(tp),
        'FN_przeoczone': int(fn),
        'FP_falszywe_alarmy': int(fp),
        'TN_poprawne': int(tn),
    })
    modele[wariant] = model
    progi_wybrane[wariant] = prog

podsumowanie = pd.DataFrame(wyniki).set_index('Wariant')
print('\nPOROWNANIE NA ZBIORZE TESTOWYM:')
print(podsumowanie.round(4).to_string())

out = Path('porownanie_xgboost_13_3strategie.csv')
podsumowanie.to_csv(out, encoding='utf-8-sig')
print('\nZapisano:', out)


POROWNANIE NA ZBIORZE TESTOWYM:
                        Prog_F2_OOF  AP_OOF  F2_OOF_dobrane  AP_test  ROC_AUC_test  Accuracy  Precision  Recall      F2  TP_wykryte  FN_przeoczone  FP_falszywe_alarmy  TN_poprawne
Wariant                                                                                                                                                                            
Mediana + flagi              0.1317  0.5335          0.5801   0.5703        0.8911    0.9051     0.4040  0.7439  0.6367          61             21                  90          998
NaN (XGBoost)                0.1302  0.5425          0.5940   0.5804        0.8949    0.9000     0.3885  0.7439  0.6289          61             21                  96          992
Hybryda: mediana + NaN       0.1252  0.5430          0.5881   0.5834        0.8926    0.8966     0.3789  0.7439  0.6237          61             21                 100          988

Zapisano: porownanie_xgboost_13_3strategie.csv


## 5. Jak uzyskac wynik dla pojedynczego przedsiebiorstwa?
Model zwraca estymowany wynik klasy pozytywnej, a prog podejmuje decyzje.
To nie musi byc skalibrowane prawdopodobienstwo bankructwa.


In [6]:
nazwa = 'Hybryda: mediana + NaN'
model_hybryda = modele[nazwa]
prog_hybryda = progi_wybrane[nazwa]
pierwszy_rekord = X_test.iloc[[0]]
wynik = float(model_hybryda.predict_proba(pierwszy_rekord)[0, 1])
print(f'Przykladowy score XGBoost: {wynik:.4f}')
print(f'Prog: {prog_hybryda:.4f}; sygnal zagrozenia: {int(wynik >= prog_hybryda)}')

# Opcjonalnie: zapis do dalszego wykorzystania (tylko zaufane pliki joblib):
# import joblib
# joblib.dump({'model': model_hybryda, 'prog': prog_hybryda,
#              'cechy': CECHY}, 'xgboost_13_hybryda.joblib')

Przykladowy score XGBoost: 0.0126
Prog: 0.1252; sygnal zagrozenia: 0
